# SEGMENTACION POR PATRON DE DEMANDA

**Autor: Abdias Figueredo V.**

**Version: 1.0**

**Agosto 2025**

**Cochabamba, Bolivia**

## Índice
1.  **Introducción**

2.  **Configuración Inicial**

3.  **Analisis Univariado**

4.  **Análisis de Distribuciones**

5.  **Análisis de Outliers**

7.  **Conclusiones**

## 1. INTRODUCCIÓN

Este proyecto tiene como objetivo realizar un Análisis Exploratorio de Datos (EDA) exhaustivo de los datos de ventas, productos e inventario de una empresa importadora de autopartes. La empresa maneja alrededor de 10,000 SKUs diferentes y tiene registros desde 2018 hasta la fecha actual, generados con IA simulando un sistema SAP Business One.

**Objetivos del EDA:**

*   Comprender la estructura y calidad de los datos disponibles.
*   Identificar patrones y tendencias en las ventas a lo largo del tiempo.
*   Analizar el comportamiento de los diferentes SKUs en términos de ventas y rotación de inventario.
*   Detectar posibles problemas en los datos, como valores atípicos o inconsistencias.
*   Generar hipótesis para futuros análisis y modelado predictivo.

**Tablas de Datos:**

*   **OITM (Tabla de Productos):** Contiene información detallada sobre cada producto, incluyendo su código, descripción, precio, etc.
*   **INV1 (Tabla de Ventas):** Registra las transacciones de venta, incluyendo la fecha, el SKU vendido, la cantidad, el precio, etc.
*   **OINM (Tabla de Inventario):** Contiene información sobre los movimientos de inventario, como entradas, salidas y ajustes.

---

## 2. CONFIGURACION INICIAL

### 2.1 Importación de librerías necesarias

In [1]:
# INSTALACION DE REQUIREMENTS
# Comando alternativo: pip install -r requirements.txt
#!pip install -r requirements.txt --quiet
# pip install --only-binary :all: statsforecast

In [1]:
# =============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# =============================================================================
# Manejo y procesamiento de datos
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd
import os
import scipy.stats as stats
from pandas.tseries.offsets import DateOffset

# Análisis de series temporales
# -----------------------------------------------------------------------------
from statsforecast import StatsForecast
from utilsforecast.plotting import plot_series
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Visualización
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Configuración de advertencias
# -----------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

# Configuración de estilos de visualización
# =============================================================================
# Configuración general de matplotlib
plt.style.use('classic')
plt.rcParams.update({
    'figure.figsize': (18, 7),
    'axes.facecolor': '#FFFFFF',  # Fondo blanco para mejor legibilidad
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16
})

# Configuración de seaborn
#sns.set_theme(style="whitegrid", 
#              rc={'axes.facecolor': '#FFFFFF'},
#              font_scale=1.1)

c:\Users\abdia\AppData\Local\Programs\Python\Python313\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
c:\Users\abdia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2.2 Carga de datos

In [22]:
# CARGA DE DATOS OITM
dataOITM = pd.read_parquet('../data/bronze/oitm.parquet')
dfOITM = dataOITM.copy()

In [4]:
# CARGA DE DATOS dfWeekStock
dataFactWeekSales = pd.read_parquet('../data/gold/FactWeekSales.parquet')
dfFactWeekSales = dataFactWeekSales.copy()

In [5]:
# CARGA DE DATOS dfABC
dataABC = pd.read_parquet('../data/silver/dfABC.parquet')
dfABC = dataABC.copy()

---

## 3. CALCULO DE METRICAS DE SEGMENTACION

In [10]:
def calculate_demand_metrics(df):
    """
    Calcula métricas de patrones de demanda por item:
    - CV2 (Coeficiente de Variación al cuadrado)
    - ADI (Average Demand Interval)
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame con las columnas ItemCode, StartWeek, NormalizedQuantity
        
    Retorna:
    --------
    pandas.DataFrame
        DataFrame con métricas por item: ItemCode, Mean, SD, CV2, ADI
    """
    def metrics_by_item(group):
        # Filtrar solo valores positivos de demanda
        positive_demand = group['NormalizedQuantity'][group['NormalizedQuantity'] > 0]
        
        # Calcular estadísticas básicas
        mean = round(positive_demand.mean(), 2) if len(positive_demand) > 0 else 0
        std = round(positive_demand.std(), 2) if len(positive_demand) > 0 else 0

        # Calcular CV2
        cv2 = round((std/mean)**2, 2) if mean > 0 else 0
        
        # Calcular ADI
        total_periods = len(group)
        demand_periods = len(positive_demand)
        adi = round(total_periods/demand_periods, 2) if demand_periods > 0 else float('inf')
        
        return pd.Series({
            'Mean': mean,
            'SD': std,
            'CV2': cv2,
            'ADI': adi
        })
    
    # Calcular métricas por item
    metrics = df.groupby('ItemCode').apply(metrics_by_item).reset_index()
    
    # Ordenar por ItemCode
    metrics = metrics.sort_values('ItemCode')
    
    # Mostrar algunos estadísticos descriptivos
    print("\nEstadísticas descriptivas de las métricas:")
    print(metrics.describe())
    
    return metrics

In [11]:
# Calcular métricas de demanda
dfSegmentation = calculate_demand_metrics(dfFactWeekSales)

# Mostrar primeros registros
print("\nMuestra de las métricas calculadas:")
display(dfSegmentation.head(10))


Estadísticas descriptivas de las métricas:
              Mean           SD         CV2          ADI
count  6475.000000  6475.000000  6475.00000  6475.000000
mean      8.103992     4.346006     0.35316     4.438131
std       2.143671     5.008269     0.52838     2.568219
min       4.240000     0.500000     0.00000     1.340000
25%       7.270000     1.620000     0.05000     1.610000
50%       7.640000     2.090000     0.08000     4.170000
75%       7.940000     5.070000     0.46000     6.390000
max      30.660000    29.920000     3.35000    15.230000

Muestra de las métricas calculadas:


,ItemCode,Mean,SD,CV2,ADI
0,SKU-00750,7.59,1.64,0.05,7.07
1,SKU-00751,8.20,1.95,0.06,8.61
2,SKU-00752,8.31,8.20,0.97,2.29
3,SKU-00754,7.15,2.10,0.09,10.15
4,SKU-00756,7.26,1.32,0.03,4.30
5,SKU-00757,8.67,2.54,0.09,8.80
6,SKU-00760,7.60,1.75,0.05,7.47
7,SKU-00761,7.69,1.88,0.06,7.62
8,SKU-00764,6.76,4.54,0.45,1.57
9,SKU-00765,7.69,2.31,0.09,7.62


## 4. SEGMENTACION POR PATRON DE DEMANDA

In [17]:
def classify_demand_pattern(df):
    """
    Clasifica los items según su patrón de demanda usando ADI y CV2.
    
    Criterios:
    - Erratic:      ADI ≤ 1.32 y CV2 > 0.49
    - Lumpy:        ADI > 1.32 y CV2 > 0.49
    - Smooth:       ADI ≤ 1.32 y CV2 ≤ 0.49
    - Intermittent: ADI > 1.32 y CV2 ≤ 0.49
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame con las columnas ItemCode, ADI, CV2
        
    Retorna:
    --------
    pandas.DataFrame
        DataFrame original con columna adicional 'DemandPattern'
    """
    def get_pattern(row):
        if row['ADI'] <= 1.32:
            return 'Erratic' if row['CV2'] > 0.49 else 'Smooth'
        else:
            return 'Lumpy' if row['CV2'] > 0.49 else 'Intermittent'
    
    # Añadir columna de clasificación
    result = df.copy()
    result['DemandPattern'] = result.apply(get_pattern, axis=1)
    
    # Mostrar resumen de clasificación
    pattern_counts = result['DemandPattern'].value_counts()
    total_items = len(result)
    
    print("\nResumen de Patrones de Demanda:")
    print("--------------------------------")
    for pattern, count in pattern_counts.items():
        percentage = round((count/total_items * 100), 1)  # Changed this line
        print(f"📊 {pattern}: {count:,} items ({percentage}%)")
        
    return result

In [18]:
# Aplicar clasificación
dfSegmentation = classify_demand_pattern(dfSegmentation)

# Mostrar algunos ejemplos
print("\nMuestra de items clasificados:")
display(dfSegmentation.head(10))


Resumen de Patrones de Demanda:
--------------------------------
📊 Intermittent: 5,311 items (82.0%)
📊 Lumpy: 1,164 items (18.0%)

Muestra de items clasificados:


,ItemCode,Mean,SD,CV2,ADI,DemandPattern
0,SKU-00750,7.59,1.64,0.05,7.07,Intermittent
1,SKU-00751,8.20,1.95,0.06,8.61,Intermittent
2,SKU-00752,8.31,8.20,0.97,2.29,Lumpy
3,SKU-00754,7.15,2.10,0.09,10.15,Intermittent
4,SKU-00756,7.26,1.32,0.03,4.30,Intermittent
5,SKU-00757,8.67,2.54,0.09,8.80,Intermittent
6,SKU-00760,7.60,1.75,0.05,7.47,Intermittent
7,SKU-00761,7.69,1.88,0.06,7.62,Intermittent
8,SKU-00764,6.76,4.54,0.45,1.57,Intermittent
9,SKU-00765,7.69,2.31,0.09,7.62,Intermittent


## 6. AJUSTES PARA EXPORTAR DATAFRAME

### 6.1 DIMENSION PRODUCTOS

In [24]:
DimItems=dfOITM
print("\nDimensión de Items:")
DimItems.head(10)


Dimensión de Items:


,ItemCode,ItemName,Brand,Category,CreateDate,FrozenFor,ItemGrp
0,SKU-00000,Recent bad Part,Wilson-Simpson,transform,2019-02-20 16:49:31,Y,447
1,SKU-00001,Activity call Part,Osborne Inc,synthesize,2016-10-17 15:36:27,Y,447
2,SKU-00002,Son almost Part,Galloway-Ray,synergize,2016-07-09 01:27:57,Y,447
3,SKU-00003,Decade create Part,Gutierrez Ltd,extend,2016-10-06 21:36:24,Y,447
4,SKU-00004,Hair concern Part,"Miles, Sharp and Davis",synergize,2017-12-01 12:37:55,Y,447
5,SKU-00005,Thing fly Part,"Baker, Hudson and Daniels",grow,2016-05-13 15:11:52,Y,447
6,SKU-00006,Brother seek Part,Robinson-Shannon,architect,2017-04-07 00:34:46,Y,447
7,SKU-00007,Tend success Part,Brown-Myers,productize,2016-03-01 08:43:45,Y,447
8,SKU-00008,Money region Part,Key-Reyes,target,2017-02-06 19:48:22,Y,447
9,SKU-00009,Product just Part,"Perry, Alexander and Brewer",aggregate,2017-09-29 12:00:39,Y,447


In [25]:
# MERGE DE DATAFRAMES DFWEEKSALES Y DFWEEKSALESSTOCKOUT
DimSubItems = pd.merge(
    dfSegmentation[['ItemCode','Mean','SD','CV2','ADI','DemandPattern']],
    dfABC[['ItemCode','LineTotal','Percentage','CumulativePercentage','ABC']],
    on=['ItemCode'],
    how='left'
)
DimSubItems.head(10)

,ItemCode,Mean,SD,CV2,ADI,DemandPattern,LineTotal,Percentage,CumulativePercentage,ABC
0,SKU-00750,7.59,1.64,0.05,7.07,Intermittent,13802.90,0.01,90.45,C
1,SKU-00751,8.20,1.95,0.06,8.61,Intermittent,27742.36,0.01,74.86,B
2,SKU-00752,8.31,8.20,0.97,2.29,Lumpy,28958.19,0.01,74.28,B
3,SKU-00754,7.15,2.10,0.09,10.15,Intermittent,5613.47,0.00,99.27,C
4,SKU-00756,7.26,1.32,0.03,4.30,Intermittent,2673.52,0.00,99.94,C
5,SKU-00757,8.67,2.54,0.09,8.80,Intermittent,13778.86,0.01,90.48,C
6,SKU-00760,7.60,1.75,0.05,7.47,Intermittent,4788.73,0.00,99.62,C
7,SKU-00761,7.69,1.88,0.06,7.62,Intermittent,16643.28,0.01,85.92,C
8,SKU-00764,6.76,4.54,0.45,1.57,Intermittent,58494.96,0.03,57.14,B
9,SKU-00765,7.69,2.31,0.09,7.62,Intermittent,10546.05,0.01,95.10,C


## 7. GUARDADO DE DATAFRAME

In [26]:
# GUARDAR EL DATAFRAME PROCESADO DFStock
DimSubItems.to_parquet('../data/gold/DimSubItems.parquet', index=False)

In [27]:
# GUARDAR EL DATAFRAME PROCESADO DFStock
DimItems.to_parquet('../data/gold/DimItems.parquet', index=False)

## 6. CONCLUSIONES

- Se identificaron que los productos mas vendidos son los SKUs 06074, 03543 y 03063.
- El análisis de distribución mostró que las ventas semanales no siguen una distribución normal (p-valor < 0.05 en la prueba de Shapiro-Wilk), lo que es común en datos de ventas del mundo real debido a factores externos como promociones o eventos estacionales.
- El analisis de outliers mostro que hay 2 valoes atipicos en la variable ventas semanales. Es coveniente normalizar estos datos ya que el proximo paso sera crear un modelo de pronostico.